## LLM-Integration

Dapr Agents bietet eine einheitliche Schnittstelle zur Anbindung an LLM-Inferenz-APIs. Diese Abstraktion ermöglicht es Entwicklern, ihre Agenten nahtlos in moderne Sprachmodelle für Schlussfolgerungen und Entscheidungsfindung zu integrieren. Das Framework umfasst mehrere LLM-Clients für verschiedene Anbieter und Modalitäten.

* DaprChatClient: Einheitliche API für LLM-Interaktionen über die Conversation API von Dapr mit integrierter Sicherheit (Scopes, Secrets, PII-Verschleierung), Ausfallsicherheit (Timeouts, Wiederholungsversuche, Circuit Breaker) und Observability über OpenTelemetry & Prometheus
* OpenAIChatClient: Umfassende Unterstützung für OpenAI-Modelle, einschließlich Chat, Einbettungen und Audio
* HFHubChatClient: Für Hugging Face-Modelle, die sowohl Chat als auch Einbettungen unterstützen
* NVIDIAChatClient: Für NVIDIA AI Foundation-Modelle, die lokale Inferenz und Chat unterstützen.
* ElevenLabsUnterstützung für Sprach- und Stimmfunktionen

Quelle: [Core Concepts](https://docs.dapr.io/developing-ai/dapr-agents/dapr-agents-core-concepts/)


In [ ]:
%%bash
rm -rf dapr-agents
git clone https://github.com/dapr/dapr-agents.git
cd dapr-agents/quickstarts
pip install dapr-agents python-dotenv ipywidgets

Zuerst die Funktion

In [ ]:
import asyncio
import random

from dapr_agents import tool

@tool
async def weather_func(location: str) -> str:
    """Get weather information for a specific location."""
    print(">>> TOOL CALLED <<<", location)
    temperature = random.randint(60, 80)
    return f"{location}: {temperature}F."


Dann der Agent welcher die Funktion aufruft

In [ ]:
from dapr_agents import DurableAgent
from dapr_agents.llm import DaprChatClient
from dapr_agents.workflow.runners import AgentRunner

# Nur einmal pro Kernel registrieren
if "weather_agent" not in globals():
    weather_agent = DurableAgent(
        name="WeatherAgent",
        role="Weather Assistant",
        instructions=[
            "You are a weather assistant.",
            "You MUST use the weather tool for ANY weather question.",
            "DO NOT answer from your own knowledge.",
            "If you do not call the tool, your answer is WRONG.",
        ],
        tools=[weather_func],
        llm=DaprChatClient(component_name="openai"),
    )

if "runner" not in globals():
    runner = AgentRunner()

async def ask_weather(task: str):
    return await runner.run(
        weather_agent,
        payload={"task": task},
    )

Eigentliche Ausführung

In [ ]:
result1 = await ask_weather("What is the weather in Zurich?")
result1

result2 = await ask_weather("What is the weather in Basel?")
result2

Mit Persistenz

In [ ]:
from dapr_agents import DurableAgent
from dapr_agents.llm import DaprChatClient
from dapr_agents.agents.configs import AgentMemoryConfig, AgentStateConfig
from dapr_agents.memory import ConversationDaprStateMemory
from dapr_agents.storage.daprstores.stateservice import StateStoreService
from dapr_agents.workflow.runners import AgentRunner

if "weather_agent" not in globals():
    weather_agent = DurableAgent(
        name="WeatherAgent",
        role="Weather Assistant",
        instructions=["Help users with weather information"],
        tools=[weather_func],
        llm=DaprChatClient(component_name="openai"),
        memory=AgentMemoryConfig(
            store=ConversationDaprStateMemory(store_name="agent-memory")
        ),
        state=AgentStateConfig(
            store=StateStoreService(store_name="agent-workflow")
        ),
    )

if "runner" not in globals():
    runner = AgentRunner()

print("Agent erstellt")


In [ ]:
await ask_weather("What is the weather in Zurich?")
await ask_weather("What is the weather in Basel?")
await ask_weather("What is the weather in Bern?")

---
### Weitere Beispiele

* [Quickstarts](https://docs.dapr.io/developing-ai/dapr-agents/dapr-agents-quickstarts/)